In [22]:
import sys, subprocess, pkgutil, os

def ensure_pyg():
    import torch
    if pkgutil.find_loader("torch_geometric") is not None:
        print("torch_geometric already available ✅")
        return
    ver = torch.__version__.split('+')[0]   # e.g. '2.9.0'
    cu  = torch.version.cuda               # e.g. '12.8'
    if cu is None:
        wheel_idx = f"torch-{ver}+cpu.html"
    else:
        cu_tag = "cu" + cu.replace('.', '')   # 'cu128'
        wheel_idx = f"torch-{ver}+{cu_tag}.html"
    idx_url = f"https://data.pyg.org/whl/{wheel_idx}"
    print("[INFO] Installing PyG wheels for", ver, cu, "from", idx_url)
    cmds = [
        [sys.executable, "-m", "pip", "install", "--upgrade", "pip", "setuptools", "wheel"],
        [sys.executable, "-m", "pip", "install", "pyg_lib", "torch_scatter", "torch_sparse", "torch_cluster", "torch_spline_conv", "-f", idx_url],
        [sys.executable, "-m", "pip", "install", "torch_geometric"],
    ]
    for c in cmds:
        print("->", " ".join(c))
        subprocess.check_call(c)

try:
    ensure_pyg()
    import torch_geometric
    print("✅ torch_geometric version:", torch_geometric.__version__)
except Exception as e:
    print("❌ Failed to prepare torch_geometric:", e)
    print("  - カーネルが別環境の可能性があります。上のセル出力のURL/コマンドでターミナル側に先に入れてから、")
    print("    Notebookのカーネルをそのvenvに切り替えてください（Kernel → Change Kernel）。")
    raise



torch_geometric already available ✅
✅ torch_geometric version: 2.7.0


In [23]:
from pathlib import Path

in_dir = Path("/home/yudai/Project/research/Vul_Detection/data/input")
files = sorted(in_dir.glob("*_cpg_input.pkl"))
print("Found:", len(files))
for i, f in enumerate(files, 1):
    print(f"{i}. {f.name}")
assert files, "data/input/*_cpg_input.pkl が見つかりません。"


Found: 2
1. 0_cpg_input.pkl
2. 1_cpg_input.pkl


In [24]:
# --- shim: map numpy._core.* -> numpy.core.* for unpickling ---
import sys, types, importlib
import numpy as _np

_aliases = {
    "numpy._core": "numpy.core",
    "numpy._core.numeric": "numpy.core.numeric",
    "numpy._core.fromnumeric": "numpy.core.fromnumeric",
    "numpy._core.multiarray": "numpy.core.multiarray",
    "numpy._core._multiarray_umath": "numpy.core._multiarray_umath",
    "numpy._core.records": "numpy.core.records",
}

for new, old in _aliases.items():
    try:
        mod = importlib.import_module(old)
    except Exception:
        # 一部は存在しない場合があるが、numeric と fromnumeric があれば十分なことが多い
        continue
    parent = new.rsplit(".", 1)[0]
    if parent not in sys.modules:
        sys.modules[parent] = types.ModuleType(parent)
    sys.modules[new] = mod

print("✅ NumPy shim active. Now you can pd.read_pickle(...) safely.")


✅ NumPy shim active. Now you can pd.read_pickle(...) safely.


In [25]:
import pandas as pd
from torch_geometric.data import Data

pkl = files[0]
df = pd.read_pickle(pkl)
print("Columns:", list(df.columns), "shape:", df.shape)
display(df.head(1))  # 先頭行プレビュー

assert {"input","target","func"}.issubset(df.columns), "必要列（input/target/func）がありません。"
assert isinstance(df.iloc[0]["input"], Data), "input が torch_geometric.data.Data ではありません。"


Columns: ['input', 'target', 'func'] shape: (965, 3)


,input,target,func
0,"[(x, [tensor([0., 0., 0., 0., 0., 0., 0., 0., ...",1,"glue(cirrus_bitblt_rop_fwd_, ROP_NAME)(CirrusV..."


In [26]:
import torch

sample = df.iloc[0]
d: Data = sample["input"]
print("target:", sample["target"])
print("func preview:", (sample["func"] or "").replace("\n"," ")[:200], "...")

print("\n[GRAPH SCHEMA]")
print("x:", tuple(d.x.shape), d.x.dtype)
print("edge_index:", tuple(d.edge_index.shape), d.edge_index.dtype)
print("num_nodes (attr):", getattr(d, "num_nodes", None))
print("num_edges:", int(d.edge_index.size(1)))

if hasattr(d, "edge_type") and d.edge_type is not None and d.edge_type.numel() == d.edge_index.size(1):
    uniq, counts = torch.unique(d.edge_type, return_counts=True)
    print("edge_type counts:", dict(zip([int(u) for u in uniq.tolist()], [int(c) for c in counts.tolist()])))
else:
    print("edge_type: None or length mismatch")


target: 1
func preview: glue(cirrus_bitblt_rop_fwd_, ROP_NAME)(CirrusVGAState *s,                              uint8_t *dst,const uint8_t *src,                              int dstpitch,int srcpitch,                          ...

[GRAPH SCHEMA]
x: (500, 769) torch.float32
edge_index: (2, 12) torch.int64
num_nodes (attr): 3
num_edges: 12
edge_type counts: {0: 6, 1: 6}


In [27]:
def real_nodes(data):
    return int(getattr(data, "num_nodes", data.x.size(0)))

nn_list = [real_nodes(d) for d in df["input"]]
e_list  = [int(d.edge_index.size(1)) for d in df["input"]]

def median(arr):
    arr = sorted(arr)
    n = len(arr)
    return arr[n//2] if n else None

print(f"graphs: {len(df)}")
print(f"nodes avg/med: {sum(nn_list)/len(nn_list):.1f} / {median(nn_list)}")
print(f"edges avg/med: {sum(e_list)/len(e_list):.1f} / {median(e_list)}")
print(f"empty-edge graphs: {sum(e==0 for e in e_list)} / {len(e_list)} ({sum(e==0 for e in e_list)/len(e_list):.2%})")


graphs: 965
nodes avg/med: 60.9 / 48
edges avg/med: 354.4 / 260
empty-edge graphs: 0 / 965 (0.00%)


In [ ]:

from collections import Counter
import torch

cnt = Counter()
has_et = True
for d in df["input"]:
    if not (hasattr(d, "edge_type") and d.edge_type is not None and d.edge_type.numel() == d.edge_index.size(1)):
        has_et = False
        break

if has_et:
    for d in df["input"]:
        if d.edge_index.numel() == 0: continue
        uniq, counts = torch.unique(d.edge_type, return_counts=True)
        for u, c in zip(uniq.tolist(), counts.tolist()):
            cnt[int(u)] += int(c)
    print("edge_type total counts:", dict(cnt))
else:
    print("edge_type not present or length mismatch; skip.")

edge_type total counts: {0: 213884, 1: 114124, 2: 13992}


In [29]:
bad = 0
for g in df["input"]:
    N = int(getattr(g, "num_nodes", g.x.size(0)))
    if g.edge_index.numel() == 0: 
        continue
    mx = int(g.edge_index.max())
    if mx >= N:
        bad += 1
print("edge_index out-of-range graphs:", bad)


edge_index out-of-range graphs: 0
